# AFD calcium imaging tracker: manual two-channel ROI tracking

This notebook tracks manually selected ROIs in the split green and red channels of a TIFF movie.

Main workflow:

1. Set `ANALYZE_CHANNELS`, ROI diameters, and `save_output` in Cell 2.
2. Load the TIFF. A 60%-through-movie frame is used as the default ROI-selection frame.
3. Preview 20 evenly spaced frames in a contact sheet, then enter the best ImageJ frame for drawing ROIs.
4. Select ROIs manually from a side-by-side green/red view. Toolbar zoom and pan clicks are ignored, so defining a zoom window will not create an ROI.
5. Track each channel independently. Green and red ROIs do not need to match, and each channel can have different ROI counts and fluorescence patterns.
6. Export or display raw ROI intensity values only. dF/F, percent dF/F, candidate detection, confidence plots, and tracked-frame QC overlays are intentionally removed.
7. Display one tracking movie with the selected channel(s), using outline-only ROI circles.


In [ ]:
# ============================================================
# CELL 1: INSTALL PACKAGES, IMPORT PACKAGES, SET INLINE PLOTTING
# ============================================================

import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tifffile": "tifffile",
    "skimage": "scikit-image",
    "tqdm": "tqdm",
}

missing_packages = []
for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(pip_name)

if len(missing_packages) > 0:
    print("Installing missing packages:")
    print(missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Package installation finished.")
    print("IMPORTANT: restart the kernel after this cell if anything was newly installed.")
else:
    print("All required packages are already installed.")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.widgets import Button
from IPython.display import HTML, display

from tifffile import TiffFile
from skimage.draw import disk
from skimage.feature import match_template
from skimage.filters import gaussian
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200

print("Current matplotlib backend:", matplotlib.get_backend())


In [ ]:
# ============================================================
# CELL 2: USER PARAMETERS AND SETTINGS
# ============================================================

TIFF_PATH = "/Volumes/DCR_6/Sandeep/CalciumImaging/20260528_mEXP1512_AIY_GCamP/20260528_mEXP1512_AIY_GCamP_rep_4/20260528_mEXP1512_AIY_GCamP_rep_4_MMStack_Pos0.ome.tif"
OUTPUT_DIR = "/Volumes/DCR_6/Sandeep/CalciumImaging/20260528_mEXP1512_AIY_GCamP/20260528_mEXP1512_AIY_GCamP_rep_4"

# Default is False so the notebook does not write files unless you choose to.
save_output = False

# Choose "green", "red", or "both".
ANALYZE_CHANNELS = "both"

FPS = 4.0
N_FRAMES_TO_ANALYZE = None

# The ROI-selection frame is chosen after loading the TIFF.
# 0.60 means ImageJ frame 600 for a 1000-frame TIFF; Cell 5 lets you override it after previewing frames.
ROI_SELECTION_FRAME_FRACTION = 0.60

# Lightweight preview for choosing the ROI drawing frame.
PREVIEW_FRAME_COUNT = 20
PREVIEW_DOWNSAMPLE = 2
PREVIEW_FPS = 2

# The TIFF is expected to be one full-width image with green on the left and red on the right.
FULL_FRAME_WIDTH = 512
SPLIT_X = FULL_FRAME_WIDTH // 2

STIM_FRAMES_IMAGEJ = [1475]
STIM_FRAMES = [f - 1 for f in STIM_FRAMES_IMAGEJ]

START_FRAME_IMAGEJ = 1
END_FRAME_IMAGEJ = None
START_FRAME = START_FRAME_IMAGEJ - 1

# All green ROIs use this radius, and all red ROIs use this radius.
# Green and red are allowed to be different.
GREEN_ROI_RADIUS = 4
RED_ROI_RADIUS = 6
GREEN_ROI_DIAMETER = 2 * GREEN_ROI_RADIUS
RED_ROI_DIAMETER = 2 * RED_ROI_RADIUS

# Tracking settings.
# SEARCH_RADIUS is the frame-to-frame movement neighborhood. The actual crop also
# includes TEMPLATE_RADIUS so the template can move by SEARCH_RADIUS pixels.
TEMPLATE_RADIUS = 10
SEARCH_RADIUS = 18
GAUSSIAN_SIGMA_FOR_TRACKING = 1.0
MAX_DISPLACEMENT_ALLOWED = 15
MIN_TEMPLATE_MATCH_SCORE = None  # Set to a number like 0.4 only if you want stricter rejection.
HOLD_POSITION_ON_BAD_TRACK = True
UPDATE_TEMPLATE_IF_GOOD = False

# Brightest-pixel snapping is off because it can jump to a nearby wrong neuron.
RECENTER_ON_BRIGHTEST_PIXEL = False
RECENTER_SEARCH_RADIUS = None  # Not used unless RECENTER_ON_BRIGHTEST_PIXEL is True.
RECENTER_GAUSSIAN_SIGMA = 0.75

# Manual ROI selection display.
# "tk" opens only the clickable ROI selector in an external window.
# The notebook switches back to inline plotting after ROI selection.
MANUAL_CLICK_BACKEND = "tk"
ALLOW_TEXT_ROI_FALLBACK = True

# Static grid settings for coordinate reading.
MAJOR_TICK_STEP = 50
MINOR_TICK_STEP = 10

CHANNEL_LABELS = {
    "green": "Green / GCaMP (left half)",
    "red": "Red / mCherry (right half)",
}
CHANNEL_MOVIE_TITLES = {
    "green": "Green",
    "red": "Red",
}

# Display-only brightness/contrast settings for preview, ROI selection, and movies.
# These do not change the raw intensity values used for analysis.
# Lower DISPLAY_HIGH_PERCENTILE and lower DISPLAY_GAMMA make dim structures easier to see.
# DISPLAY_OUTPUT_FLOOR lifts true-black pixels slightly, like raising display brightness.
DISPLAY_LOW_PERCENTILE = {
    "green": 0.0,
    "red": 0.0,
}
DISPLAY_HIGH_PERCENTILE = {
    "green": 97.5,
    "red": 99.4,
}
DISPLAY_GAMMA = {
    "green": 0.25,
    "red": 0.70,
}
DISPLAY_OUTPUT_FLOOR = {
    "green": 0.12,
    "red": 0.03,
}
CHANNEL_COLORS = {
    "green": "lime",
    "red": "tomato",
}
CHANNEL_ROI_DIAMETERS = {
    "green": float(GREEN_ROI_DIAMETER),
    "red": float(RED_ROI_DIAMETER),
}
CHANNEL_ROI_RADII = {channel: diameter / 2 for channel, diameter in CHANNEL_ROI_DIAMETERS.items()}


def parse_active_channels(channel_request):
    value = str(channel_request).strip().lower()
    if value in {"both", "green+red", "red+green", "all"}:
        return ["green", "red"]
    if value in {"green", "g"}:
        return ["green"]
    if value in {"red", "r"}:
        return ["red"]
    raise ValueError('ANALYZE_CHANNELS must be "green", "red", or "both".')


ACTIVE_CHANNELS = parse_active_channels(ANALYZE_CHANNELS)

for channel, diameter in CHANNEL_ROI_DIAMETERS.items():
    if diameter <= 0:
        raise ValueError(f"{channel} ROI diameter must be positive.")

if save_output:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

print(
    f"Setup: channels={ACTIVE_CHANNELS}, "
    f"green radius={GREEN_ROI_RADIUS}px, red radius={RED_ROI_RADIUS}px, "
    f"save_output={save_output}"
)


In [ ]:
# ============================================================
# CELL 3: LOAD TIFF LAZILY
# ============================================================

tif = TiffFile(TIFF_PATH)
series = tif.series[0]
movie = series.asarray(out="memmap")
movie = np.squeeze(movie)

if movie.ndim != 3:
    raise ValueError(f"Expected movie shape [frames, height, width], but got {movie.shape}")

n_total_frames, frame_h, frame_w = movie.shape

if frame_w != FULL_FRAME_WIDTH:
    print(f"Warning: expected frame width {FULL_FRAME_WIDTH}, got {frame_w}; using midpoint split.")
    SPLIT_X = frame_w // 2
else:
    SPLIT_X = FULL_FRAME_WIDTH // 2

if N_FRAMES_TO_ANALYZE is None:
    n_frames = n_total_frames
else:
    n_frames = min(N_FRAMES_TO_ANALYZE, n_total_frames)

if END_FRAME_IMAGEJ is None:
    END_FRAME = n_frames - 1
else:
    END_FRAME = min(END_FRAME_IMAGEJ - 1, n_frames - 1)

if START_FRAME < 0 or START_FRAME > END_FRAME:
    raise ValueError("START_FRAME_IMAGEJ and END_FRAME_IMAGEJ define an invalid analysis range.")

ROI_SELECTION_FRAME_IMAGEJ = int(round(n_frames * ROI_SELECTION_FRAME_FRACTION))
ROI_SELECTION_FRAME_IMAGEJ = int(np.clip(ROI_SELECTION_FRAME_IMAGEJ, 1, n_frames))
ROI_SELECTION_FRAME = ROI_SELECTION_FRAME_IMAGEJ - 1

print(
    f"Loaded {n_total_frames} frames ({frame_h} x {frame_w}); "
    f"analyzing ImageJ frames {START_FRAME + 1}-{END_FRAME + 1}; "
    f"default ROI frame: ImageJ {ROI_SELECTION_FRAME_IMAGEJ}."
)


In [ ]:
# ============================================================
# CELL 4: BASIC IMAGE, DISPLAY, AND ROI FUNCTIONS
# ============================================================


def get_frame(frame_idx):
    return movie[frame_idx].astype(np.float32)


def split_channels(frame):
    green = frame[:, :SPLIT_X]
    red = frame[:, SPLIT_X:]
    return {"green": green, "red": red}


def get_channel_frame(frame_idx, channel):
    channel = str(channel).lower()
    if channel not in {"green", "red"}:
        raise ValueError("channel must be 'green' or 'red'")
    return split_channels(get_frame(frame_idx))[channel]


def get_green_frame(frame_idx):
    return get_channel_frame(frame_idx, "green")


def get_red_frame(frame_idx):
    return get_channel_frame(frame_idx, "red")


def display_setting(setting, channel, default):
    if isinstance(setting, dict):
        return setting.get(channel, default)
    return setting


def normalize_for_display(img, channel=None, p_low=None, p_high=None, gamma=None, output_floor=None):
    if p_low is None:
        p_low = display_setting(DISPLAY_LOW_PERCENTILE, channel, 0.0)
    if p_high is None:
        p_high = display_setting(DISPLAY_HIGH_PERCENTILE, channel, 99.5)
    if gamma is None:
        gamma = display_setting(DISPLAY_GAMMA, channel, 1.0)
    if output_floor is None:
        output_floor = display_setting(DISPLAY_OUTPUT_FLOOR, channel, 0.0)

    lo, hi = np.nanpercentile(img, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(img, dtype=np.float32)

    out = (img.astype(np.float32) - lo) / (hi - lo)
    out = np.clip(out, 0, 1)

    if gamma is not None and gamma > 0 and gamma != 1:
        out = np.power(out, gamma)

    if output_floor is not None and output_floor > 0:
        output_floor = float(np.clip(output_floor, 0, 0.95))
        out = output_floor + (1 - output_floor) * out

    return out


def add_coordinate_grid(ax, img_shape, channel=None):
    h, w = img_shape
    ax.set_xlim(0, w - 1)
    ax.set_ylim(h - 1, 0)
    ax.set_xticks(np.arange(0, w, MAJOR_TICK_STEP))
    ax.set_yticks(np.arange(0, h, MAJOR_TICK_STEP))
    ax.set_xticks(np.arange(0, w, MINOR_TICK_STEP), minor=True)
    ax.set_yticks(np.arange(0, h, MINOR_TICK_STEP), minor=True)
    ax.grid(which="major", linewidth=0.7, alpha=0.45)
    ax.grid(which="minor", linewidth=0.3, alpha=0.20)
    if channel is None:
        ax.set_xlabel("x coordinate")
    else:
        ax.set_xlabel(f"x coordinate in {channel} channel")
    ax.set_ylabel("y coordinate")


def roi_diameter_for_channel(channel):
    return CHANNEL_ROI_DIAMETERS[channel]


def roi_radius_for_channel(channel):
    return CHANNEL_ROI_RADII[channel]


def roi_label(channel, roi_id):
    return f"{channel[0].upper()}{int(roi_id)}"


def validate_channel_coordinate(channel, x, y):
    img = get_channel_frame(ROI_SELECTION_FRAME, channel)
    h, w = img.shape
    if not (0 <= x < w):
        print(f"Warning: {channel} x={x:.2f} is outside channel range 0 to {w - 1}.")
    if not (0 <= y < h):
        print(f"Warning: {channel} y={y:.2f} is outside image height 0 to {h - 1}.")


def parse_xy_string(s):
    parts = s.replace(" ", "").split(",")
    if len(parts) != 2:
        raise ValueError("Please enter coordinates as x,y")
    return float(parts[0]), float(parts[1])


def parse_many_xy_string(s):
    points = []
    for chunk in s.split(";"):
        chunk = chunk.strip()
        if len(chunk) > 0:
            points.append(parse_xy_string(chunk))
    return points


def circular_mask_values(img, x, y, diameter):
    radius = float(diameter) / 2
    rr, cc = disk((float(y), float(x)), radius, shape=img.shape)
    return img[rr, cc]


def extract_raw_roi_intensity(channel_img, x, y, diameter):
    roi_vals = circular_mask_values(channel_img, x, y, diameter)
    raw_mean = float(np.nanmean(roi_vals)) if len(roi_vals) > 0 else np.nan
    raw_median = float(np.nanmedian(roi_vals)) if len(roi_vals) > 0 else np.nan
    return raw_mean, raw_median, int(len(roi_vals))


def show_green_and_red(frame_idx, title_extra=""):
    channels = split_channels(get_frame(frame_idx))
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, channel in zip(axes, ["green", "red"]):
        img = channels[channel]
        ax.imshow(normalize_for_display(img, channel=channel), cmap="gray")
        title = f"{CHANNEL_LABELS[channel]}\nImageJ frame {frame_idx + 1}"
        if title_extra:
            title += "\n" + title_extra
        ax.set_title(title)
        add_coordinate_grid(ax, img.shape, channel=channel)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# CELL 5: PREVIEW 20 EVENLY SPACED FRAMES
# ============================================================
#
# This cell only displays a lightweight contact sheet.
# Choose your preferred ImageJ frame number from the titles,
# then enter it in the next cell.
# ============================================================


def get_preview_frame_image(frame_idx, channel):
    img = get_channel_frame(frame_idx, channel)
    if PREVIEW_DOWNSAMPLE > 1:
        img = img[::PREVIEW_DOWNSAMPLE, ::PREVIEW_DOWNSAMPLE]
    return normalize_for_display(img, channel=channel)


def make_preview_panel(frame_idx):
    green = get_preview_frame_image(frame_idx, "green")
    red = get_preview_frame_image(frame_idx, "red")
    return np.concatenate([green, red], axis=1)


def show_roi_frame_preview_contact_sheet():
    global PREVIEW_IMAGEJ_FRAMES

    PREVIEW_IMAGEJ_FRAMES = np.unique(
        np.round(np.linspace(1, n_frames, PREVIEW_FRAME_COUNT)).astype(int)
    )
    PREVIEW_IMAGEJ_FRAMES = PREVIEW_IMAGEJ_FRAMES[
        (PREVIEW_IMAGEJ_FRAMES >= 1) & (PREVIEW_IMAGEJ_FRAMES <= n_frames)
    ]
    preview_python_frames = PREVIEW_IMAGEJ_FRAMES - 1

    n_cols = 5
    n_rows = int(np.ceil(len(preview_python_frames) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows), squeeze=False)

    for ax in axes.ravel():
        ax.set_axis_off()

    for ax, imagej_frame, frame_idx in zip(axes.ravel(), PREVIEW_IMAGEJ_FRAMES, preview_python_frames):
        ax.imshow(make_preview_panel(int(frame_idx)), cmap="gray", vmin=0, vmax=1, aspect="equal")
        ax.set_title(f"ImageJ {int(imagej_frame)}", fontsize=10)

    fig.suptitle("Preview frames for ROI selection: green left, red right", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    print("Preview ImageJ frames:", [int(x) for x in PREVIEW_IMAGEJ_FRAMES])
    print("Default ROI selection frame:", int(ROI_SELECTION_FRAME_IMAGEJ))


show_roi_frame_preview_contact_sheet()


In [ ]:
# ============================================================
# CELL 6: CHOOSE ROI SELECTION FRAME
# ============================================================
#
# Set ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE to one of the previewed ImageJ frame numbers,
# or leave it as None and answer the prompt.
# ============================================================

ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE = None


def choose_roi_selection_frame():
    global ROI_SELECTION_FRAME_IMAGEJ, ROI_SELECTION_FRAME

    default_frame = int(ROI_SELECTION_FRAME_IMAGEJ)

    if ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE is None:
        user_text = input(
            f"Enter ImageJ frame for ROI selection, or press Enter for default {default_frame}: "
        ).strip()
        if len(user_text) > 0:
            chosen_frame = int(user_text)
        else:
            chosen_frame = default_frame
    else:
        chosen_frame = int(ROI_SELECTION_FRAME_IMAGEJ_OVERRIDE)

    ROI_SELECTION_FRAME_IMAGEJ = int(np.clip(chosen_frame, 1, n_frames))
    ROI_SELECTION_FRAME = ROI_SELECTION_FRAME_IMAGEJ - 1

    print(f"ROI selection frame set to ImageJ {ROI_SELECTION_FRAME_IMAGEJ}.")


choose_roi_selection_frame()


In [ ]:
# ============================================================
# CELL 7: MANUAL SIDE-BY-SIDE ROI SELECTION
# ============================================================
#
# Use the external interactive matplotlib window to click ROI centers.
# Green and red are displayed side by side so you can choose corresponding fields of view.
# Toolbar zoom/pan clicks are ignored and will not create ROIs.
# Use the Home button to return both channels to the full view, then zoom elsewhere.
# Press Done or ENTER/RETURN when finished. Later plots are restored to inline notebook output.
# ============================================================

ROI_TABLE_COLUMNS = [
    "channel",
    "roi_id",
    "x_initial",
    "y_initial",
    "frame_initial_python",
    "frame_initial_imagej",
    "roi_diameter",
    "roi_radius",
    "source",
]


def make_roi_tables_from_points(points_by_channel, frame_idx, source="manual_click"):
    roi_tables = {}
    for channel in ACTIVE_CHANNELS:
        rows = []
        for i, (x, y) in enumerate(points_by_channel.get(channel, []), start=1):
            validate_channel_coordinate(channel, x, y)
            rows.append({
                "channel": channel,
                "roi_id": int(i),
                "x_initial": float(x),
                "y_initial": float(y),
                "frame_initial_python": int(frame_idx),
                "frame_initial_imagej": int(frame_idx + 1),
                "roi_diameter": float(roi_diameter_for_channel(channel)),
                "roi_radius": float(roi_radius_for_channel(channel)),
                "source": source,
            })
        roi_tables[channel] = pd.DataFrame(rows, columns=ROI_TABLE_COLUMNS)
    return roi_tables


def toolbar_is_active(fig):
    toolbar = getattr(fig.canvas, "toolbar", None)
    mode = getattr(toolbar, "mode", "") if toolbar is not None else ""
    if mode is None:
        return False
    mode_text = str(mode)
    return mode_text not in {"", "None", "_Mode.NONE"}


def select_rois_side_by_side(frame_idx, backend="tk"):
    channels_to_show = ["green", "red"]
    selected_points = {channel: [] for channel in ACTIVE_CHANNELS}
    artist_history = []

    try:
        get_ipython().run_line_magic("matplotlib", backend)
    except Exception as err:
        print("Could not switch matplotlib backend.")
        print("Error:", err)

    done = {"value": False}
    buttons = []

    fig, axes = plt.subplots(1, 2, figsize=(9, 8))
    plt.subplots_adjust(bottom=0.16, wspace=0.22)

    axes_by_channel = {}
    axis_to_channel = {}
    full_limits = {}

    for ax, channel in zip(axes, channels_to_show):
        img = get_channel_frame(frame_idx, channel)
        display_img = normalize_for_display(img, channel=channel)
        ax.imshow(display_img, cmap="gray", vmin=0, vmax=1, aspect="equal")
        ax.set_title(
            f"{CHANNEL_MOVIE_TITLES[channel]}\nImageJ frame {frame_idx + 1}",
            fontsize=11,
        )
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(0, img.shape[1] - 1)
        ax.set_ylim(img.shape[0] - 1, 0)
        ax.set_xlabel(f"x coordinate in {channel} channel")
        ax.set_ylabel("y coordinate")
        axes_by_channel[channel] = ax
        axis_to_channel[ax] = channel
        full_limits[channel] = ((0, img.shape[1] - 1), (img.shape[0] - 1, 0))

    def redraw():
        fig.canvas.draw_idle()

    def draw_roi(channel, x, y):
        ax = axes_by_channel[channel]
        roi_id = len(selected_points[channel])
        radius = roi_radius_for_channel(channel)
        color = CHANNEL_COLORS[channel]

        circle = plt.Circle(
            (x, y),
            radius,
            fill=False,
            linewidth=1.7,
            edgecolor=color,
        )
        ax.add_patch(circle)

        label = ax.text(
            x + radius + 2,
            y,
            roi_label(channel, roi_id),
            color=color,
            fontsize=9,
            bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1),
        )

        artist_history.append({"channel": channel, "artists": [circle, label]})
        redraw()

    def add_roi(channel, x, y):
        selected_points[channel].append((float(x), float(y)))
        draw_roi(channel, float(x), float(y))

    def undo_last(event=None):
        if len(artist_history) == 0:
            return
        entry = artist_history.pop()
        channel = entry["channel"]
        if len(selected_points[channel]) > 0:
            selected_points[channel].pop()
        for artist in entry["artists"]:
            artist.remove()
        redraw()

    def clear_all(event=None):
        while len(artist_history) > 0:
            entry = artist_history.pop()
            for artist in entry["artists"]:
                artist.remove()
        for channel in selected_points:
            selected_points[channel] = []
        redraw()

    def home_view(event=None):
        for channel, ax in axes_by_channel.items():
            xlim, ylim = full_limits[channel]
            ax.set_xlim(*xlim)
            ax.set_ylim(*ylim)
        redraw()

    def finish(event=None):
        done["value"] = True

    def on_click(event):
        if event.button != 1:
            return
        if event.inaxes not in axis_to_channel:
            return
        if event.xdata is None or event.ydata is None:
            return
        if toolbar_is_active(fig):
            return

        channel = axis_to_channel[event.inaxes]
        if channel not in ACTIVE_CHANNELS:
            return

        add_roi(channel, event.xdata, event.ydata)

    def on_key(event):
        if event.key in {"enter", "return"}:
            finish(event)
        elif event.key in {"backspace", "delete", "u"}:
            undo_last(event)
        elif event.key in {"h", "home"}:
            home_view(event)

    def on_close(event):
        done["value"] = True

    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("key_press_event", on_key)
    fig.canvas.mpl_connect("close_event", on_close)

    button_specs = [
        ("Home", [0.26, 0.04, 0.11, 0.05], home_view),
        ("Undo", [0.39, 0.04, 0.11, 0.05], undo_last),
        ("Clear", [0.52, 0.04, 0.11, 0.05], clear_all),
        ("Done", [0.65, 0.04, 0.11, 0.05], finish),
    ]
    for label, rect, callback in button_specs:
        button_ax = fig.add_axes(rect)
        button = Button(button_ax, label)
        button.on_clicked(callback)
        buttons.append(button)

    try:
        plt.show(block=False)
        while (not done["value"]) and plt.fignum_exists(fig.number):
            fig.canvas.flush_events()
            time.sleep(0.05)
    except Exception as err:
        print("Manual clicking did not work in this environment.")
        print("Error:", err)
    finally:
        try:
            fig.canvas.flush_events()
        except Exception:
            pass

    return make_roi_tables_from_points(selected_points, frame_idx, source="manual_click")


def enter_rois_by_text(frame_idx):
    points_by_channel = {channel: [] for channel in ACTIVE_CHANNELS}
    print("Text fallback: enter ROI centers as x,y; x,y. Leave blank for no ROIs in a channel.")
    for channel in ACTIVE_CHANNELS:
        text = input(f"Enter {channel} ROI centers: ").strip()
        if len(text) > 0:
            points_by_channel[channel] = parse_many_xy_string(text)
    return make_roi_tables_from_points(points_by_channel, frame_idx, source="manual_text")


roi_tables = select_rois_side_by_side(
    ROI_SELECTION_FRAME,
    backend=MANUAL_CLICK_BACKEND,
)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

total_selected = sum(len(df) for df in roi_tables.values())
if total_selected == 0 and ALLOW_TEXT_ROI_FALLBACK:
    roi_tables = enter_rois_by_text(ROI_SELECTION_FRAME)
    total_selected = sum(len(df) for df in roi_tables.values())

if total_selected == 0:
    raise RuntimeError("No ROIs were selected.")

roi_counts = {channel: len(roi_tables[channel]) for channel in ACTIVE_CHANNELS}
print("Selected ROIs:", roi_counts)

if save_output:
    for channel in ACTIVE_CHANNELS:
        out_path = os.path.join(OUTPUT_DIR, f"selected_{channel}_rois.csv")
        roi_tables[channel].to_csv(out_path, index=False)


In [ ]:
# ============================================================
# CELL 8: VISUALIZE SELECTED ROIS
# ============================================================


def plot_selected_rois_side_by_side(frame_idx, roi_tables):
    fig, axes = plt.subplots(1, 2, figsize=(9, 8))

    for ax, channel in zip(axes, ["green", "red"]):
        img = get_channel_frame(frame_idx, channel)
        ax.imshow(normalize_for_display(img, channel=channel), cmap="gray", vmin=0, vmax=1, aspect="equal")
        ax.set_title(f"{CHANNEL_MOVIE_TITLES[channel]}\nImageJ frame {frame_idx + 1}")
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(0, img.shape[1] - 1)
        ax.set_ylim(img.shape[0] - 1, 0)
        ax.set_xlabel(f"x coordinate in {channel} channel")
        ax.set_ylabel("y coordinate")

        df = roi_tables.get(channel, pd.DataFrame())
        if len(df) == 0:
            continue

        for _, row in df.iterrows():
            roi_id = int(row["roi_id"])
            x = float(row["x_initial"])
            y = float(row["y_initial"])
            radius = float(row["roi_radius"])
            color = CHANNEL_COLORS[channel]

            circle = plt.Circle(
                (x, y),
                radius,
                fill=False,
                linewidth=1.7,
                edgecolor=color,
            )
            ax.add_patch(circle)
            ax.text(
                x + radius + 2,
                y,
                roi_label(channel, roi_id),
                color=color,
                fontsize=9,
                bbox=dict(facecolor="black", alpha=0.55, edgecolor="none", pad=1),
            )

    plt.tight_layout()
    plt.show()


plot_selected_rois_side_by_side(ROI_SELECTION_FRAME, roi_tables)


In [ ]:
# ============================================================
# CELL 9: TRACKING HELPER FUNCTIONS
# ============================================================


def crop_with_bounds(img, x, y, radius):
    h, w = img.shape
    x = int(round(x))
    y = int(round(y))
    x0 = max(0, x - radius)
    x1 = min(w, x + radius + 1)
    y0 = max(0, y - radius)
    y1 = min(h, y + radius + 1)
    crop = img[y0:y1, x0:x1]
    return crop, x0, x1, y0, y1


def make_template(channel_img, x, y, template_radius):
    smooth = gaussian(channel_img, sigma=GAUSSIAN_SIGMA_FOR_TRACKING, preserve_range=True)
    template, x0, x1, y0, y1 = crop_with_bounds(smooth, x, y, template_radius)
    return template


def track_one_step(channel_img, prev_x, prev_y, template):
    smooth = gaussian(channel_img, sigma=GAUSSIAN_SIGMA_FOR_TRACKING, preserve_range=True)

    search_crop_radius = int(np.ceil(TEMPLATE_RADIUS + SEARCH_RADIUS))
    search_crop, sx0, sx1, sy0, sy1 = crop_with_bounds(smooth, prev_x, prev_y, search_crop_radius)

    if search_crop.shape[0] < template.shape[0] or search_crop.shape[1] < template.shape[1]:
        return float(prev_x), float(prev_y), False

    result = match_template(search_crop, template, pad_input=False)
    if not np.any(np.isfinite(result)):
        return float(prev_x), float(prev_y), False

    result_for_peak = np.where(np.isfinite(result), result, -np.inf)
    y_peak, x_peak = np.unravel_index(np.argmax(result_for_peak), result_for_peak.shape)
    match_score = float(result_for_peak[y_peak, x_peak])

    template_h, template_w = template.shape
    new_x = sx0 + x_peak + (template_w - 1) / 2
    new_y = sy0 + y_peak + (template_h - 1) / 2

    displacement = float(np.sqrt((new_x - prev_x) ** 2 + (new_y - prev_y) ** 2))
    accepted = displacement <= MAX_DISPLACEMENT_ALLOWED
    if MIN_TEMPLATE_MATCH_SCORE is not None:
        accepted = accepted and match_score >= MIN_TEMPLATE_MATCH_SCORE

    return float(new_x), float(new_y), bool(accepted)

def recenter_on_brightest_pixel(channel_img, x, y, search_radius):
    if search_radius is None or search_radius <= 0:
        return float(x), float(y)

    crop, x0, x1, y0, y1 = crop_with_bounds(channel_img, x, y, int(np.ceil(search_radius)))
    if crop.size == 0:
        return float(x), float(y)

    if RECENTER_GAUSSIAN_SIGMA and RECENTER_GAUSSIAN_SIGMA > 0:
        search_img = gaussian(crop, sigma=RECENTER_GAUSSIAN_SIGMA, preserve_range=True)
    else:
        search_img = crop

    yy, xx = np.indices(search_img.shape)
    cx = float(x) - x0
    cy = float(y) - y0
    circular_search_mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= float(search_radius) ** 2
    search_img = np.where(circular_search_mask & np.isfinite(search_img), search_img, -np.inf)

    if not np.any(np.isfinite(search_img)):
        return float(x), float(y)

    y_peak, x_peak = np.unravel_index(np.argmax(search_img), search_img.shape)
    return float(x0 + x_peak), float(y0 + y_peak)


In [ ]:
# ============================================================
# CELL 10: BUILD INITIAL ROI POSITIONS FOR TRACKING
# ============================================================

INITIAL_POSITION_COLUMNS = [
    "channel",
    "roi_id",
    "frame",
    "frame_imagej",
    "x",
    "y",
    "roi_diameter",
    "roi_radius",
    "source",
]


def build_initial_positions_df(roi_tables):
    rows = []

    for channel, roi_df in roi_tables.items():
        if len(roi_df) == 0:
            continue
        for _, row in roi_df.iterrows():
            rows.append({
                "channel": channel,
                "roi_id": int(row["roi_id"]),
                "frame": int(row["frame_initial_python"]),
                "frame_imagej": int(row["frame_initial_imagej"]),
                "x": float(row["x_initial"]),
                "y": float(row["y_initial"]),
                "roi_diameter": float(row["roi_diameter"]),
                "roi_radius": float(row["roi_radius"]),
                "source": "initial",
            })

    initial_positions_df = pd.DataFrame(rows, columns=INITIAL_POSITION_COLUMNS)
    if len(initial_positions_df) == 0:
        raise RuntimeError("No ROI positions were found.")

    initial_positions_df = initial_positions_df.sort_values(["channel", "roi_id"]).reset_index(drop=True)

    for _, row in initial_positions_df.iterrows():
        validate_channel_coordinate(row["channel"], row["x"], row["y"])

    if save_output:
        initial_positions_path = os.path.join(OUTPUT_DIR, "initial_roi_positions.csv")
        initial_positions_df.to_csv(initial_positions_path, index=False)

    return initial_positions_df


initial_positions_df = build_initial_positions_df(roi_tables)


In [ ]:
# ============================================================
# CELL 11: CHANNEL-INDEPENDENT ROI TRACKING FUNCTIONS
# ============================================================

TRACE_COLUMNS = [
    "channel",
    "frame",
    "frame_imagej",
    "time_sec",
    "roi_id",
    "x",
    "y",
    "roi_diameter",
    "roi_radius",
    "raw_mean",
    "raw_median",
    "n_pixels",
    "tracking_direction",
    "segment_start_frame",
    "segment_start_imagej",
]


def track_roi_segment(channel, roi_id, start_frame, end_frame, start_x, start_y, roi_diameter, direction):
    if direction not in [+1, -1]:
        raise ValueError("direction must be +1 or -1")

    if direction == +1 and end_frame < start_frame:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    if direction == -1 and end_frame > start_frame:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    start_img = get_channel_frame(start_frame, channel)
    if RECENTER_ON_BRIGHTEST_PIXEL:
        search_radius = RECENTER_SEARCH_RADIUS
        if search_radius is None:
            search_radius = roi_diameter / 2
        start_x, start_y = recenter_on_brightest_pixel(
            start_img,
            start_x,
            start_y,
            search_radius,
        )

    template = make_template(start_img, start_x, start_y, TEMPLATE_RADIUS)

    current_x = float(start_x)
    current_y = float(start_y)
    rows = []
    frame_range = range(start_frame, end_frame + direction, direction)

    for frame_idx in frame_range:
        channel_img = get_channel_frame(frame_idx, channel)

        if frame_idx != start_frame:
            x_new, y_new, accepted = track_one_step(channel_img, current_x, current_y, template)

            if (not accepted) and HOLD_POSITION_ON_BAD_TRACK:
                x_pred = current_x
                y_pred = current_y
            else:
                x_pred = x_new
                y_pred = y_new

            current_x = float(x_pred)
            current_y = float(y_pred)

            if UPDATE_TEMPLATE_IF_GOOD and accepted:
                template = make_template(channel_img, current_x, current_y, TEMPLATE_RADIUS)

        if RECENTER_ON_BRIGHTEST_PIXEL:
            search_radius = RECENTER_SEARCH_RADIUS
            if search_radius is None:
                search_radius = roi_diameter / 2
            current_x, current_y = recenter_on_brightest_pixel(
                channel_img,
                current_x,
                current_y,
                search_radius,
            )

        raw_mean, raw_median, n_pixels = extract_raw_roi_intensity(
            channel_img,
            current_x,
            current_y,
            roi_diameter,
        )

        rows.append({
            "channel": channel,
            "frame": int(frame_idx),
            "frame_imagej": int(frame_idx + 1),
            "time_sec": float(frame_idx / FPS),
            "roi_id": int(roi_id),
            "x": float(current_x),
            "y": float(current_y),
            "roi_diameter": float(roi_diameter),
            "roi_radius": float(roi_diameter / 2),
            "raw_mean": raw_mean,
            "raw_median": raw_median,
            "n_pixels": int(n_pixels),
            "tracking_direction": "forward" if direction == +1 else "backward",
            "segment_start_frame": int(start_frame),
            "segment_start_imagej": int(start_frame + 1),
        })

    return pd.DataFrame(rows, columns=TRACE_COLUMNS)


def track_single_roi(initial_row, start_frame, end_frame):
    channel = str(initial_row["channel"])
    roi_id = int(initial_row["roi_id"])
    initial_frame = int(initial_row["frame"])
    initial_x = float(initial_row["x"])
    initial_y = float(initial_row["y"])
    roi_diameter = float(initial_row["roi_diameter"])
    segments = []

    if initial_frame >= start_frame:
        segments.append(track_roi_segment(
            channel=channel,
            roi_id=roi_id,
            start_frame=initial_frame,
            end_frame=start_frame,
            start_x=initial_x,
            start_y=initial_y,
            roi_diameter=roi_diameter,
            direction=-1,
        ))

    if end_frame >= initial_frame:
        segments.append(track_roi_segment(
            channel=channel,
            roi_id=roi_id,
            start_frame=initial_frame,
            end_frame=end_frame,
            start_x=initial_x,
            start_y=initial_y,
            roi_diameter=roi_diameter,
            direction=+1,
        ))

    roi_trace = pd.concat(segments, ignore_index=True)
    roi_trace = roi_trace[
        (roi_trace["frame"] >= start_frame) &
        (roi_trace["frame"] <= end_frame)
    ]
    roi_trace = (
        roi_trace
        .sort_values(["channel", "roi_id", "frame", "tracking_direction"])
        .drop_duplicates(subset=["channel", "roi_id", "frame"], keep="last")
        .sort_values("frame")
        .reset_index(drop=True)
    )

    return roi_trace


def track_all_rois(initial_positions_df, start_frame, end_frame):
    all_traces = []

    for _, initial_row in tqdm(initial_positions_df.iterrows(), total=len(initial_positions_df), desc="Tracking ROIs"):
        all_traces.append(track_single_roi(initial_row, start_frame=start_frame, end_frame=end_frame))

    if len(all_traces) == 0:
        return pd.DataFrame(columns=TRACE_COLUMNS)

    traces_df = pd.concat(all_traces, ignore_index=True)
    traces_df = traces_df.sort_values(["channel", "roi_id", "frame"]).reset_index(drop=True)
    return traces_df


In [ ]:
# ============================================================
# CELL 12: RUN TRACKING AND PREPARE RAW ROI INTENSITY TABLES
# ============================================================

traces_df = track_all_rois(
    initial_positions_df=initial_positions_df,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
)

channel_traces = {
    channel: traces_df[traces_df["channel"] == channel].copy().reset_index(drop=True)
    for channel in ACTIVE_CHANNELS
}


def make_wide_raw_table(channel_df, channel, value_col="raw_mean"):
    if len(channel_df) == 0:
        return pd.DataFrame()

    times = (
        channel_df[["frame_imagej", "time_sec"]]
        .drop_duplicates()
        .sort_values("frame_imagej")
    )
    wide = channel_df.pivot(index="frame_imagej", columns="roi_id", values=value_col)
    wide.columns = [roi_label(channel, c) for c in wide.columns]
    wide = wide.reset_index()
    wide = times.merge(wide, on="frame_imagej", how="left")
    return wide


raw_mean_tables = {}
raw_median_tables = {}

for channel in ACTIVE_CHANNELS:
    channel_df = channel_traces[channel]
    raw_mean_tables[channel] = make_wide_raw_table(channel_df, channel, value_col="raw_mean")
    raw_median_tables[channel] = make_wide_raw_table(channel_df, channel, value_col="raw_median")

    if save_output and len(channel_df) > 0:
        long_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_roi_traces_long.csv")
        mean_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_mean_by_roi.csv")
        median_path = os.path.join(OUTPUT_DIR, f"{channel}_raw_median_by_roi.csv")

        channel_df.to_csv(long_path, index=False)
        raw_mean_tables[channel].to_csv(mean_path, index=False)
        raw_median_tables[channel].to_csv(median_path, index=False)

print("Tracking complete. Raw tables are in raw_mean_tables and raw_median_tables.")


In [ ]:
# ============================================================
# CELL 13: OPTIONAL RAW INTENSITY TRACE PLOTS
# ============================================================
#
# This plots raw mean intensity only. dF/F and percent dF/F are not calculated.
# Set PLOT_RAW_TRACES = False if you only want the tables from Cell 10.
# ============================================================

PLOT_RAW_TRACES = True


def plot_raw_traces(channel_df, channel, value_col="raw_mean", stim_frames=None):
    if len(channel_df) == 0:
        return

    roi_ids = sorted(channel_df["roi_id"].unique())
    fig, ax = plt.subplots(figsize=(11, 4.5))

    for roi_id in roi_ids:
        sub = channel_df[channel_df["roi_id"] == roi_id]
        ax.plot(sub["time_sec"], sub[value_col], label=roi_label(channel, roi_id), linewidth=1)

    if stim_frames is not None:
        for sf in stim_frames:
            ax.axvline(sf / FPS, linestyle="--", linewidth=1)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel(value_col)
    ax.set_title(f"{CHANNEL_MOVIE_TITLES[channel]} raw ROI intensity")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


if PLOT_RAW_TRACES:
    for channel in ACTIVE_CHANNELS:
        plot_raw_traces(channel_traces[channel], channel, value_col="raw_mean", stim_frames=STIM_FRAMES)


In [ ]:
# ============================================================
# CELL 14: IMAGESC-STYLE RAW INTENSITY HEATMAPS
# ============================================================
#
# These are additional heatmap views of the same raw mean intensity traces.
# Each row is one ROI, and each column is one time point/frame.
# ============================================================

PLOT_RAW_IMAGESC = True
IMAGESC_VALUE_COL = "raw_mean"
IMAGESC_LOW_PERCENTILE = 1
IMAGESC_HIGH_PERCENTILE = 99
IMAGESC_CMAP = "viridis"


def make_trace_matrix(channel_df, channel, value_col="raw_mean"):
    if len(channel_df) == 0:
        return np.empty((0, 0)), [], np.array([])

    frames = np.array(sorted(channel_df["frame"].unique()))
    times = (
        channel_df[["frame", "time_sec"]]
        .drop_duplicates()
        .set_index("frame")
        .loc[frames, "time_sec"]
        .values
    )
    roi_ids = sorted(channel_df["roi_id"].unique())
    matrix = np.full((len(roi_ids), len(frames)), np.nan, dtype=float)

    frame_to_col = {frame: idx for idx, frame in enumerate(frames)}
    roi_to_row = {roi_id: idx for idx, roi_id in enumerate(roi_ids)}

    for _, row in channel_df.iterrows():
        matrix[roi_to_row[row["roi_id"]], frame_to_col[row["frame"]]] = row[value_col]

    labels = [roi_label(channel, roi_id) for roi_id in roi_ids]
    return matrix, labels, times


def plot_raw_imagesc(channel_df, channel, value_col="raw_mean", stim_frames=None):
    matrix, labels, times = make_trace_matrix(channel_df, channel, value_col=value_col)
    if matrix.size == 0:
        return

    finite_values = matrix[np.isfinite(matrix)]
    if len(finite_values) == 0:
        return

    vmin, vmax = np.nanpercentile(finite_values, [IMAGESC_LOW_PERCENTILE, IMAGESC_HIGH_PERCENTILE])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = np.nanmin(finite_values), np.nanmax(finite_values)

    fig_height = max(2.8, 0.45 * len(labels) + 1.8)
    fig, ax = plt.subplots(figsize=(11, fig_height))

    extent = [times[0], times[-1], len(labels) + 0.5, 0.5]
    im = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        cmap=IMAGESC_CMAP,
        vmin=vmin,
        vmax=vmax,
        extent=extent,
    )

    if stim_frames is not None:
        for sf in stim_frames:
            ax.axvline(sf / FPS, linestyle="--", linewidth=1, color="white", alpha=0.9)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("ROI")
    ax.set_yticks(np.arange(1, len(labels) + 1))
    ax.set_yticklabels(labels)
    ax.set_title(f"{CHANNEL_MOVIE_TITLES[channel]} raw ROI intensity imagesc")

    cbar = plt.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label(value_col)

    plt.tight_layout()
    plt.show()


if PLOT_RAW_IMAGESC:
    for channel in ACTIVE_CHANNELS:
        plot_raw_imagesc(channel_traces[channel], channel, value_col=IMAGESC_VALUE_COL, stim_frames=STIM_FRAMES)


In [ ]:
# ============================================================
# CELL 15: INTERACTIVE TRACKING MOVIE WITH OUTLINE-ONLY ROIS
# ============================================================
#
# If both green and red were analyzed, this shows both channels in one movie.
# ROI overlays are circles only, without center dots.
# Optional MP4 saving only happens when both SAVE_MP4 and save_output are True.
# ============================================================

mpl.rcParams["animation.embed_limit"] = 300  # MB

# ---------------- USER SETTINGS ----------------
MOVIE_FRAME_STEP = 10
MOVIE_FPS = 5
MOVIE_DPI = 150

MOVIE_START_FRAME_IMAGEJ = 1
MOVIE_END_FRAME_IMAGEJ = n_frames

CROP_TO_TRACK_REGION = True
CROP_MARGIN = 30

SHOW_TRAIL = False
TRAIL_LENGTH_FRAMES = 200

SAVE_MP4 = False
MP4_FILENAME = f"roi_tracking_movie_every_{MOVIE_FRAME_STEP}_frames.mp4"
# ------------------------------------------------

movie_channels = [
    channel for channel in ACTIVE_CHANNELS
    if channel in channel_traces and len(channel_traces[channel]) > 0
]

if len(movie_channels) == 0:
    raise RuntimeError("No tracked channels are available for the movie.")

movie_start = max(0, MOVIE_START_FRAME_IMAGEJ - 1)
movie_end = min(n_frames - 1, MOVIE_END_FRAME_IMAGEJ - 1)
frame_list = list(range(movie_start, movie_end + 1, MOVIE_FRAME_STEP))

if len(frame_list) == 0:
    raise ValueError("No frames selected for movie. Check frame range settings.")

channel_height, channel_width = get_channel_frame(movie_start, movie_channels[0]).shape

if CROP_TO_TRACK_REGION:
    crop_source = pd.concat([channel_traces[channel] for channel in movie_channels], ignore_index=True)
    x_all = crop_source["x"].values
    y_all = crop_source["y"].values

    x_min = max(0, int(np.floor(np.nanmin(x_all) - CROP_MARGIN)))
    x_max = min(channel_width - 1, int(np.ceil(np.nanmax(x_all) + CROP_MARGIN)))

    y_min = max(0, int(np.floor(np.nanmin(y_all) - CROP_MARGIN)))
    y_max = min(channel_height - 1, int(np.ceil(np.nanmax(y_all) + CROP_MARGIN)))
else:
    x_min, x_max = 0, channel_width - 1
    y_min, y_max = 0, channel_height - 1


def get_display_channel_crop(frame_idx, channel):
    img = get_channel_frame(frame_idx, channel)
    crop = img[y_min:y_max + 1, x_min:x_max + 1]
    return normalize_for_display(crop, channel=channel)


first_frame = frame_list[0]
n_cols = len(movie_channels)
fig, axes = plt.subplots(1, n_cols, figsize=(5.5 * n_cols, 6), squeeze=False)
axes = axes[0]

image_artists = {}
title_artists = {}
circle_artists = {}
text_artists = {}
trail_artists = {}
roi_ids_by_channel = {}

for ax, channel in zip(axes, movie_channels):
    first_img = get_display_channel_crop(first_frame, channel)
    image_artists[channel] = ax.imshow(first_img, cmap="gray", vmin=0, vmax=1, aspect="equal")
    title_artists[channel] = ax.set_title(CHANNEL_MOVIE_TITLES[channel], fontsize=12)
    ax.set_xlim(0, first_img.shape[1] - 1)
    ax.set_ylim(first_img.shape[0] - 1, 0)
    ax.set_xlabel("x coordinate, cropped view")
    ax.set_ylabel("y coordinate")

    roi_ids = sorted(channel_traces[channel]["roi_id"].unique())
    roi_ids_by_channel[channel] = roi_ids
    circle_artists[channel] = {}
    text_artists[channel] = {}
    trail_artists[channel] = {}

    cmap = plt.get_cmap("tab10")
    for k, roi_id in enumerate(roi_ids):
        color = cmap(k % 10)
        radius = roi_radius_for_channel(channel)

        circ = plt.Circle(
            (0, 0),
            radius,
            fill=False,
            lw=1.8,
            color=color,
            visible=False,
        )
        ax.add_patch(circ)

        label_text = ax.text(
            0,
            0,
            roi_label(channel, roi_id),
            color=color,
            fontsize=8,
            visible=False,
            bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1),
        )

        trail_line, = ax.plot(
            [],
            [],
            "-",
            lw=1.0,
            color=color,
            alpha=0.75,
            visible=False,
        )

        circle_artists[channel][roi_id] = circ
        text_artists[channel][roi_id] = label_text
        trail_artists[channel][roi_id] = trail_line

fig_title = fig.suptitle("", fontsize=12, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])


def update_movie(frame_i):
    frame_idx = frame_list[frame_i]
    fig_title.set_text(f"ImageJ frame {frame_idx + 1} / {n_frames}")

    artists = [fig_title]

    for channel in movie_channels:
        image_artists[channel].set_data(get_display_channel_crop(frame_idx, channel))
        title_artists[channel].set_text(CHANNEL_MOVIE_TITLES[channel])
        artists.extend([image_artists[channel], title_artists[channel]])

        sub_now = channel_traces[channel][channel_traces[channel]["frame"] == frame_idx].copy()

        for roi_id in roi_ids_by_channel[channel]:
            circ = circle_artists[channel][roi_id]
            label_text = text_artists[channel][roi_id]
            trail_line = trail_artists[channel][roi_id]

            sub_roi = sub_now[sub_now["roi_id"] == roi_id]
            if len(sub_roi) == 0:
                circ.set_visible(False)
                label_text.set_visible(False)
                trail_line.set_visible(False)
                continue

            x = float(sub_roi["x"].iloc[0])
            y = float(sub_roi["y"].iloc[0])
            radius = roi_radius_for_channel(channel)

            xc = x - x_min
            yc = y - y_min

            circ.center = (xc, yc)
            circ.set_visible(True)

            label_text.set_position((xc + radius + 2, yc))
            label_text.set_text(roi_label(channel, roi_id))
            label_text.set_visible(True)

            if SHOW_TRAIL:
                sub_hist = channel_traces[channel][
                    (channel_traces[channel]["roi_id"] == roi_id) &
                    (channel_traces[channel]["frame"] <= frame_idx) &
                    (channel_traces[channel]["frame"] >= frame_idx - TRAIL_LENGTH_FRAMES)
                ].copy().sort_values("frame")

                if len(sub_hist) >= 2:
                    xh = sub_hist["x"].values - x_min
                    yh = sub_hist["y"].values - y_min
                    trail_line.set_data(xh, yh)
                    trail_line.set_visible(True)
                else:
                    trail_line.set_visible(False)
            else:
                trail_line.set_visible(False)

            artists.extend([circ, label_text, trail_line])

    return artists


anim_roi_tracking = FuncAnimation(
    fig,
    update_movie,
    frames=len(frame_list),
    interval=1000 / MOVIE_FPS,
    blit=False,
    cache_frame_data=False,
)

if SAVE_MP4:
    if save_output:
        movie_output_dir = Path(OUTPUT_DIR)
        movie_output_dir.mkdir(parents=True, exist_ok=True)
        mp4_path = movie_output_dir / MP4_FILENAME
        writer = FFMpegWriter(fps=MOVIE_FPS)
        anim_roi_tracking.save(mp4_path, writer=writer, dpi=MOVIE_DPI)
        print("Saved mp4:", mp4_path)
    else:
        print("SAVE_MP4 is True, but save_output is False. Set save_output = True in Cell 2 to write the movie file.")

plt.close(fig)
display(HTML(anim_roi_tracking.to_jshtml()))
